<a href="https://colab.research.google.com/github/Madhaveffai/pytorch-learning/blob/main/week2/Training_Loop_Pattern.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

train()vs eval()mode

In [11]:
import torch
import torch.nn as nn

model = nn.Sequential(nn.Linear(1,16), nn.ReLU(), nn.Linear(16, 1))

model.train()
model.eval()

Sequential(
  (0): Linear(in_features=1, out_features=16, bias=True)
  (1): ReLU()
  (2): Linear(in_features=16, out_features=1, bias=True)
)

Full training loop with validation

In [15]:
from torch.utils.data import DataLoader, TensorDataset, random_split

X = torch.linspace(-3, 3, 400).unsqueeze(1)
y = torch.sin(X) + 0.1 * torch.randn(400,1)

dataset = TensorDataset(X, y)
train_set, val_set = random_split(dataset, [320,80])
train_loader = DataLoader(train_set, batch_size=32, shuffle=32)
val_loader = DataLoader(val_set, batch_size=32, shuffle=False)

class MLP(nn.Module):
  def __init__(self):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(1, 32), nn.ReLU(),
        nn.Linear(32, 32), nn.ReLU(),
        nn.Linear(32, 1)
    )
  def forward(self, x):
    return self.net(x)

model = MLP()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

Loop

In [17]:
for epoch in range(50):
  model.train()
  train_loss = 0
  for X_batch, y_batch in train_loader:
    y_pred = model(X_batch)
    loss = loss_fn(y_pred, y_batch)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    train_loss += loss.item()
  train_loss /= len(train_loader)

  model.eval()
  val_loss = 0
  with torch.no_grad():
    for X_batch, y_batch in val_loader:
      y_pred = model(X_batch)
      val_loss += loss_fn(y_pred, y_batch).item()
  val_loss /= len(val_loader)

  if epoch % 10 == 0:
    print(f"Epoch {epoch} | Train: {train_loss:.4f} | Val: {val_loss:.4f}")

Epoch 0 | Train: 0.2695 | Val: 0.1273
Epoch 10 | Train: 0.0131 | Val: 0.0096
Epoch 20 | Train: 0.0119 | Val: 0.0109
Epoch 30 | Train: 0.0099 | Val: 0.0094
Epoch 40 | Train: 0.0104 | Val: 0.0101
